# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, following best practices for working with [Croissant](https://mlcommons.org/croissant/) schemas.

### Dataset Source

The dataset Croissant schema is available at: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

We first load the dataset metadata and find the available record sets using `mlcroissant`. This will help us understand the structure before extracting any data.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access high-level information
meta = dataset.metadata
print("Dataset name:", meta.name)
print("Description:", meta.description)
print("Version:", getattr(meta, 'version', 'N/A'))


## 2. Data Overview

Let's identify available record sets (`@id` values), the fields (columns) in each, and their unique identifiers. All references use their `@id` for clarity and reproducibility.

Below, we print the IDs and names of each record set in the dataset, then preview the first five records from the main record set.

In [ ]:
# List all record sets and their info by @id
record_sets_info = getattr(meta, "record_set", [])  # Croissant Python metadata convention
if not record_sets_info:
    # Try singular (schema may use 'recordSet')
    record_sets_info = getattr(meta, "recordSet", [])
    # ToList if dict
    if isinstance(record_sets_info, dict):
        record_sets_info = [record_sets_info]

record_set_ids = []
print("Available Record Sets and their @id:")
for rs in record_sets_info:
    if hasattr(rs, "@id"):
        id_ = getattr(rs, "@id")
        name = getattr(rs, "name", id_)
        print(f"- {name} | @id: {id_}")
        record_set_ids.append(id_)

# Fallback: if record sets are not directly present, check distribution, which is common in Croissant
if not record_set_ids:
    print("No explicit record sets found in Croissant metadata. Attempting to infer from distribution...")
    dist = getattr(meta, 'distribution', [])
    if isinstance(dist, dict):
        dist = [dist]
    for d in dist:
        id_ = getattr(d, '@id', None)
        print(f"- Distribution @id: {id_}")
        record_set_ids.append(id_)

# For this dataset, use the first found record set/distribution as main (per Croissant structure)
main_record_set_id = record_set_ids[0] if record_set_ids else None

print(f"\nShowing first 5 records from main record set/distribution (@id: {main_record_set_id}):")
try:
    for idx, record in enumerate(dataset.records(record_set=main_record_set_id)):
        print(json.dumps(record, indent=2))
        if idx >= 4:
            break
except Exception as e:
    print("Could not iterate records (may be metadata-only or file require download):", e)

## 3. Data Extraction

We extract data from all available record sets (or, if not present, the available distribution), referencing each entity by their `@id`. The extracted data for each is loaded as a pandas DataFrame for further analysis.

In [ ]:
# Extract records for each available record set/distribution using @id
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            print(f"Loaded {df.shape[0]} rows with {df.shape[1]} columns from '{record_set_id}'")
        else:
            df = pd.DataFrame()
            print(f"No records found in {record_set_id}")
        dataframes[record_set_id] = df
    except Exception as e:
        print(f"Could not load records from {record_set_id}: {e}")

# Print columns for the main record set
main_df = dataframes.get(main_record_set_id, pd.DataFrame())
print("\nColumns in main DataFrame:")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)

Let's perform basic EDA tasks:

- Filter records based on a numeric field (e.g., remove low abundance proteins).
- Normalize a numeric field (z-score).
- Group by a categorical field and aggregate.

Please update variable assignments below with the actual `@id` of the numeric and categorical fields in your dataset. The code prints the normalized and grouped results.

In [ ]:
# ---- FIELD SELECTION ----
# You can use the output of 'print(main_df.columns.tolist())' above to select appropriate field names (they correspond to @id values).

# For demonstration, we'll try to use typical field ids found in proteomics datasets.
# Replace below with actual fields if output above differs.
# Examples (update if schema changes):
# numeric_field_id = 'abundance_total'       # Or whatever matches your columns (the value is the @id of the field)
# group_field_id = 'sample_id'               # For grouping samples

if not main_df.empty:
    numeric_candidates = [col for col in main_df.columns if 'abund' in col.lower() or 'count' in col.lower() or 'mw' in col.lower() or main_df[col].dtype in ['float64', 'int64']]
    group_candidates = [col for col in main_df.columns if 'sample' in col.lower() or 'group' in col.lower() or 'treatment' in col.lower()]

    # Default to first candidate if present
    numeric_field_id = numeric_candidates[0] if numeric_candidates else main_df.select_dtypes(include=['number']).columns[0]
    group_field_id = group_candidates[0] if group_candidates else None

    print(f"Selected numeric field for EDA: {numeric_field_id}")
    if group_field_id:
        print(f"Selected group field for EDA: {group_field_id}")

    # Filtering
    threshold = main_df[numeric_field_id].mean()  # Use mean as threshold for demonstration
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records in '{main_record_set_id}' where {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' (z-score) for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Grouped aggregation
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data (mean) by '{group_field_id}':")
        print(grouped_df.head())
else:
    print("Main DataFrame is empty; skipping EDA.")

## 5. Visualization

Visualize numeric field distribution (e.g., using a histogram and boxplot) and relationships (e.g., between abundance and another feature).

All axes and legends use `@id` values for reference.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not main_df.empty and numeric_field_id in main_df.columns:
    plt.figure(figsize=(12, 4))
    plt.subplot(1,2,1)
    sns.histplot(main_df[numeric_field_id].dropna(), bins=40, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)

    plt.subplot(1,2,2)
    sns.boxplot(x=main_df[numeric_field_id])
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

    # If there's a group field, plot as violin- or boxplot
    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(
            x=main_df[group_field_id],
            y=main_df[numeric_field_id],
            palette='Set2')
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field found for plotting or empty DataFrame.")

## 6. Conclusion

In this notebook, we demonstrated how to load and explore the FAIR^2 dataset using the `mlcroissant` library and Croissant schema standards. We used proper entity referencing (`@id`) for all record sets, fields, and columns, and provided examples for data filtering, normalization, grouping, and visualization.

This workflow can be adapted to any FAIR dataset described via a Croissant schema and provides a reproducible starting point for scientific data analysis.

For more advanced tasks, refer to the [mlcroissant documentation](https://github.com/mlcommons/croissant) and the dataset's schema file.